<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 7 · Martes — Métricas de Clasificación</h1>
<h3>Accuracy, Precision, Recall, F1, Matriz de Confusión, ROC-AUC</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

1. Entender por qué **accuracy es engañosa** con clases desbalanceadas.
2. Dominar la **matriz de confusión** y sus 4 cuadrantes (TP, TN, FP, FN).
3. Conocer **Precision, Recall, F1-score** y cuándo usar cada una.
4. Saber qué es la **curva ROC** y el **AUC**.
5. Resolver **2 ejercicios profundos** decidiendo qué métrica es la correcta para cada problema.

# 1. ¿Por qué accuracy NO basta?

### 🧠 El problema del médico flojo

Imagina que tu modelo detecta cáncer. El 99% de los pacientes que examina están sanos, solo el 1% tiene cáncer.

**Un "modelo" trivial que SIEMPRE diga "sano" tendría 99% de accuracy.** Suena genial... hasta que te das cuenta que **no detecta ningún caso de cáncer**.

Por eso necesitamos métricas que entiendan la **asimetría del problema**: detectar un cáncer real importa MUCHO más que confundir un sano con un enfermo.

# 2. La Matriz de Confusión

Para clasificación binaria, una matriz 2x2 que muestra TODO:

|  | Predicho NEGATIVO (0) | Predicho POSITIVO (1) |
|---|---|---|
| **Real NEGATIVO (0)** | TN (True Negative) ✅ | FP (False Positive) ⚠️ |
| **Real POSITIVO (1)** | FN (False Negative) ❌ | TP (True Positive) ✅ |

**Las 4 celdas:**
- **TP (Verdadero Positivo):** dijimos enfermo y SÍ está enfermo ✅
- **TN (Verdadero Negativo):** dijimos sano y SÍ está sano ✅
- **FP (Falso Positivo):** dijimos enfermo pero estaba sano ⚠️ → "falsa alarma"
- **FN (Falso Negativo):** dijimos sano pero estaba enfermo ❌ → "se nos escapó"

### Decisión de negocio

| Contexto | ¿Qué error es PEOR? |
|---|---|
| Detector de cáncer | **FN** — perder un enfermo es fatal |
| Filtro de spam | **FP** — mandar al spam un correo importante es molesto |
| Aprobación de préstamo | **FP** — prestarle a alguien que no paga es costoso |
| Recomendar películas | Casi da igual — ningún error es grave |

# 3. Las 4 métricas que vas a usar siempre

| Métrica | Fórmula | ¿Qué responde? | Cuándo importa |
|---|---|---|---|
| **Accuracy** | (TP+TN)/total | ¿Qué % acerté en general? | Cuando las clases están balanceadas |
| **Precision** | TP/(TP+FP) | De los que predije positivos, ¿qué % lo eran de verdad? | Cuando FP es costoso (spam, créditos) |
| **Recall** | TP/(TP+FN) | De todos los positivos reales, ¿qué % capturé? | Cuando FN es costoso (cáncer, fraude) |
| **F1** | media armónica de P y R | Balance entre Precision y Recall | Cuando los dos errores cuestan algo |

### Memorizando rápido

- **Precision** → "De lo que dije que era positivo, ¿cuánto era POSITIVO de verdad?" (calidad de las alarmas)
- **Recall** → "De los positivos REALES, ¿cuántos atrapé?" (cobertura)
- **F1** → "¿Estoy bien en ambos?"

## 👀 Demo en vivo — accuracy vs F1 en clases desbalanceadas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              ConfusionMatrixDisplay, roc_curve, roc_auc_score)

# Pima Diabetes — desbalanceado (~65/35)
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
df = pd.read_csv(url, names=cols)

# Truco oculto: ceros que en realidad son nulos
for c in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
    df[c] = df[c].replace(0, np.nan)
df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=['Outcome'])
y = df['Outcome']
print(f'Distribución de clases: {y.value_counts().to_dict()}')
print(f'% positivos (diabetes): {y.mean()*100:.1f}%')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Modelo 1: DUMMY que siempre dice "sano" (la clase mayoritaria)
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)

# Modelo 2: KNN real
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11)).fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_pred_knn   = knn.predict(X_test)

filas = []
for nombre, pred in [('DummyClassifier (siempre dice sana)', y_pred_dummy), ('KNN (k=11)', y_pred_knn)]:
    filas.append({
        'Modelo': nombre,
        'Accuracy':  accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall':    recall_score(y_test, pred, zero_division=0),
        'F1':        f1_score(y_test, pred, zero_division=0)
    })

tabla = pd.DataFrame(filas).set_index('Modelo').round(4)
print(tabla)
print('\n🚨 El Dummy tiene accuracy ~65% pero Precision=0, Recall=0, F1=0.')
print('   Eso significa que NO detecta ningún caso de diabetes. INÚTIL.')
print('   Si solo miráramos accuracy, parecería un "modelo decente".')

In [ ]:
# Matriz de confusión del KNN — visualizada
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dummy, ax=axes[0], cmap='Blues',
                                         display_labels=['Sana', 'Diabetes'])
axes[0].set_title('Dummy (todo "sana")')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn, ax=axes[1], cmap='Greens',
                                         display_labels=['Sana', 'Diabetes'])
axes[1].set_title('KNN (k=11)')
plt.tight_layout(); plt.show()

# classification_report — todo en uno
print('Classification report del KNN:')
print(classification_report(y_test, y_pred_knn, target_names=['Sana', 'Diabetes']))

# 4. Curva ROC y AUC — más allá de un umbral

Los clasificadores no solo dan una clase, dan **probabilidades**. Por defecto, sklearn corta en 0.5 (>0.5 → positivo). Pero **podemos mover ese umbral**.

- **Umbral bajo** (ej. 0.3) → más casos predichos positivos → más Recall, menos Precision.
- **Umbral alto** (ej. 0.7) → menos casos predichos positivos → más Precision, menos Recall.

**La curva ROC** muestra cómo cambia el rendimiento del modelo al mover el umbral.

**El AUC (Area Under Curve)** resume la curva en un solo número:
- 1.0 = clasificador perfecto
- 0.5 = clasificador aleatorio (tirar moneda)
- < 0.5 = peor que aleatorio (raro)

In [ ]:
# Calcular ROC del KNN
y_proba = knn.predict_proba(X_test)[:, 1]  # probabilidad de la clase positiva
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'KNN (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('Curva ROC — KNN sobre Pima Diabetes', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.show()

print(f'AUC = {auc:.3f} → cuánto mejor que tirar una moneda')

---
# 🏋️ Ejercicios profundos

## 🩺 Ejercicio 1 — Métricas en Pima Diabetes

Vamos a profundizar en el problema de la diabetes que vimos ayer, pero ahora ENFOCADOS EN MÉTRICAS.

**Setup:** mismo dataset Pima Indians Diabetes (limpieza de ceros incluida).

**Parte A — Baseline crítico**
1. Crear 3 modelos:
   - `DummyClassifier(strategy='most_frequent')` — el flojo
   - `DummyClassifier(strategy='stratified')` — el aleatorio respetando proporciones
   - `KNeighborsClassifier(n_neighbors=11)` con StandardScaler
2. Reportar **accuracy, precision, recall, F1** de los 3 en una tabla.
3. Comentar: ¿el Dummy 'most_frequent' es "bueno" según accuracy? ¿Por qué F1 lo desenmascara?

**Parte B — Matriz de confusión a fondo**
4. Para el KNN, mostrar la matriz de confusión visualizada.
5. **Calcular a mano:**
   - ¿Cuántos pacientes con diabetes el modelo dijo que estaban sanos? (FN)
   - Sobre el total de pacientes con diabetes en el test, ¿qué porcentaje se le escapó al modelo? (1 - Recall)
6. Si este modelo se usara en un hospital real, ¿lo aprobarías? Justifica.

**Parte C — Mover el umbral para mejorar Recall**
7. Obtener las probabilidades con `predict_proba()`.
8. Calcular las predicciones para 3 umbrales: 0.3, 0.5 (default), 0.7.
9. Reportar Precision y Recall para cada umbral.
10. ¿Qué pasa con FN cuando bajamos el umbral a 0.3?

**Parte D — Curva ROC + AUC**
11. Graficar la curva ROC del KNN.
12. Calcular el AUC e interpretarlo en una frase.

In [ ]:
# Parte A — 3 modelos + tabla de métricas 👇



In [ ]:
# Parte B — Matriz de confusión + análisis crítico 👇



In [ ]:
# Parte C + D — Mover umbral + curva ROC 👇



## 💰 Ejercicio 2 — Detección de billetes falsos (Banknote)

Ahora un dataset binario más LIMPIO. Trabajas en seguridad bancaria. Te dan 1,372 mediciones de billetes (transformadas) y debes detectar los **falsos**.

**Dataset:**
```python
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/banknote_authentication.csv'
cols = ['variance', 'skewness', 'curtosis', 'entropy', 'class']  # class: 0=genuino, 1=falso
df = pd.read_csv(url, names=cols)
```

**Parte A — Exploración + decisión de negocio**
1. ¿Cuál es la proporción de clases? ¿Está balanceado?
2. **Pregunta crítica:** en este problema, ¿qué error es PEOR?
   - **FP:** decir falso a un billete genuino → cliente molesto.
   - **FN:** decir genuino a un billete falso → el banco pierde dinero.
3. Argumentar qué métrica vas a OPTIMIZAR.

**Parte B — Pipeline + tuning**
4. Train/test 80/20 con stratify.
5. Pipeline con StandardScaler + KNN.
6. GridSearchCV sobre `n_neighbors=[3, 5, 7, 11, 15]` y `weights=['uniform', 'distance']`.
7. **Usar `scoring='f1'` en GridSearch** (no accuracy).

**Parte C — Reporte completo de métricas**
8. Para el mejor modelo, mostrar:
   - `classification_report` completo.
   - Matriz de confusión visualizada.
   - Curva ROC con AUC.

**Parte D — Comparación con umbral ajustado**
9. Ajustar el umbral para que el modelo tenga **Recall ≥ 0.95** (no se nos puede escapar ningún falso).
10. ¿Cuánto baja la Precision? ¿Es aceptable para el banco?

**Parte E — Decisión final**
11. Resumen: ¿qué modelo + umbral entregarías? Justifica desde el negocio.

In [ ]:
# Parte A — Exploración + decisión de negocio 👇



In [ ]:
# Parte B — Pipeline + GridSearchCV con scoring='f1' 👇



In [ ]:
# Parte C + D + E — Reporte completo + umbral + decisión 👇



---
## 📌 Cierre del día

- ✅ **Accuracy es engañosa** con clases desbalanceadas — un dummy puede tener buena accuracy y ser inútil.
- ✅ La **matriz de confusión** muestra TP, TN, FP, FN — el primer paso siempre.
- ✅ **Precision:** calidad de las alarmas (importa cuando FP cuesta).
- ✅ **Recall:** cobertura (importa cuando FN cuesta — cáncer, fraude).
- ✅ **F1:** balance entre ambos.
- ✅ **ROC-AUC:** rendimiento global, independiente del umbral.
- ✅ Podemos **mover el umbral** para sacrificar precision en favor de recall (o viceversa).

### 🔜 Mañana — Miércoles 3 de junio

Decision Tree Classifier y Random Forest Classifier. Modelos no-lineales que dominan en problemas tabulares de clasificación + Feature Importance para explicar al negocio.